In [1]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
text = """Shall I compare thee to a summer's day? Thou art more lovely and more temperate.
Rough winds do shake the darling buds of May, And summer's lease hath all too short a date."""

In [3]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

In [4]:
print(tokenizer.word_index)

{'a': 1, "summer's": 2, 'more': 3, 'and': 4, 'shall': 5, 'i': 6, 'compare': 7, 'thee': 8, 'to': 9, 'day': 10, 'thou': 11, 'art': 12, 'lovely': 13, 'temperate': 14, 'rough': 15, 'winds': 16, 'do': 17, 'shake': 18, 'the': 19, 'darling': 20, 'buds': 21, 'of': 22, 'may': 23, 'lease': 24, 'hath': 25, 'all': 26, 'too': 27, 'short': 28, 'date': 29}


In [5]:
total_words = len(tokenizer.word_index) + 1
print("Total words:", total_words)
# adding one more word so that we can effectively add the padding to it

Total words: 30


In [6]:
# convert the text into sequence 

input_sequence = []

for line in text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):

        n_gram_sequence = token_list[:i+1]
        input_sequence.append(n_gram_sequence)

input_sequence

# we are teacing the model 
# [5,6] --> means input 5 and predicts 6 more like if u enter shall it have to predict i
# [5,6,7] --> input as shall & I it have to predict the compare

[[5, 6],
 [5, 6, 7],
 [5, 6, 7, 8],
 [5, 6, 7, 8, 9],
 [5, 6, 7, 8, 9, 1],
 [5, 6, 7, 8, 9, 1, 2],
 [5, 6, 7, 8, 9, 1, 2, 10],
 [5, 6, 7, 8, 9, 1, 2, 10, 11],
 [5, 6, 7, 8, 9, 1, 2, 10, 11, 12],
 [5, 6, 7, 8, 9, 1, 2, 10, 11, 12, 3],
 [5, 6, 7, 8, 9, 1, 2, 10, 11, 12, 3, 13],
 [5, 6, 7, 8, 9, 1, 2, 10, 11, 12, 3, 13, 4],
 [5, 6, 7, 8, 9, 1, 2, 10, 11, 12, 3, 13, 4, 3],
 [5, 6, 7, 8, 9, 1, 2, 10, 11, 12, 3, 13, 4, 3, 14],
 [15, 16],
 [15, 16, 17],
 [15, 16, 17, 18],
 [15, 16, 17, 18, 19],
 [15, 16, 17, 18, 19, 20],
 [15, 16, 17, 18, 19, 20, 21],
 [15, 16, 17, 18, 19, 20, 21, 22],
 [15, 16, 17, 18, 19, 20, 21, 22, 23],
 [15, 16, 17, 18, 19, 20, 21, 22, 23, 4],
 [15, 16, 17, 18, 19, 20, 21, 22, 23, 4, 2],
 [15, 16, 17, 18, 19, 20, 21, 22, 23, 4, 2, 24],
 [15, 16, 17, 18, 19, 20, 21, 22, 23, 4, 2, 24, 25],
 [15, 16, 17, 18, 19, 20, 21, 22, 23, 4, 2, 24, 25, 26],
 [15, 16, 17, 18, 19, 20, 21, 22, 23, 4, 2, 24, 25, 26, 27],
 [15, 16, 17, 18, 19, 20, 21, 22, 23, 4, 2, 24, 25, 26, 27, 28],
 [1

In [7]:
# to get the max sequence 
# so that we can use it in padding 

max_sequence_len = max(
    [len(x) for x in input_sequence]
)

max_sequence_len

18

In [8]:
input_sequence = np.array(
    pad_sequences(
        input_sequence,
        padding='pre',
        maxlen=max_sequence_len
    )
)

In [9]:
# now lets seperate the input and output 

x = input_sequence[:, :-1] #takes every entity except the last
y = input_sequence[:, -1] #takes last entity alone 

# [5,6,7]--> x =[5,6], y=[7]

In [10]:
x.shape,y.shape

((31, 17), (31,))

In [11]:
# Convert y into One-Hot Encoding
y = tf.keras.utils.to_categorical(
    y,
    num_classes=total_words
)

print("y shape:", y.shape)

# currently y have only [1,2,3.....]
# using one hot encoding [1,0,0...]
#                        [0,1,0....]

y shape: (31, 30)


In [13]:
with tf.device('/GPU:0'):
    model = Sequential([
        Embedding(
            total_words,
            64,
            input_length=max_sequence_len - 1
        ),
        LSTM(64),
        Dense(
            total_words,
            activation='softmax'
        )
    ])

    model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 17, 64)            1920      
                                                                 
 lstm_1 (LSTM)               (None, 64)                33024     
                                                                 
 dense_1 (Dense)             (None, 30)                1950      
                                                                 
Total params: 36,894
Trainable params: 36,894
Non-trainable params: 0
_________________________________________________________________


In [15]:
history = model.fit(
    x,
    y,
    epochs=100,
    verbose=1
)

Epoch 1/100
1/1 [==============================] - 5s 5s/step - loss: 3.4025 - accuracy: 0.0000e+00
Epoch 2/100
1/1 [==============================] - 0s 12ms/step - loss: 3.3981 - accuracy: 0.0000e+00
Epoch 3/100
1/1 [==============================] - 0s 12ms/step - loss: 3.3937 - accuracy: 0.0323
Epoch 4/100
1/1 [==============================] - 0s 13ms/step - loss: 3.3893 - accuracy: 0.1935
Epoch 5/100
1/1 [==============================] - 0s 13ms/step - loss: 3.3848 - accuracy: 0.1935
Epoch 6/100
1/1 [==============================] - 0s 14ms/step - loss: 3.3802 - accuracy: 0.1935
Epoch 7/100
1/1 [==============================] - 0s 18ms/step - loss: 3.3755 - accuracy: 0.1613
Epoch 8/100
1/1 [==============================] - 0s 15ms/step - loss: 3.3705 - accuracy: 0.1290
Epoch 9/100
1/1 [==============================] - 0s 13ms/step - loss: 3.3652 - accuracy: 0.1290
Epoch 10/100
1/1 [==============================] - 0s 12ms/step - loss: 3.3595 - accuracy: 0.1290
Epoch 11/100


In [16]:
def generate_text(seed_text, next_words, max_sequence_len):

    for _ in range(next_words):

        token_list = tokenizer.texts_to_sequences(
            [seed_text]
        )[0]

        token_list = pad_sequences(
            [token_list],
            maxlen=max_sequence_len - 1,
            padding='pre'
        )

        predicted = np.argmax(
            model.predict(
                token_list,
                verbose=0
            ),
            axis=1
        )[0]

        output_word = ""

        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break

        seed_text += " " + output_word

    return seed_text

In [17]:
# now doing the prediction 
seed_text = "Shall I compare thee"
next_words = 10

generated_text = generate_text(
    seed_text,
    next_words,
    max_sequence_len
)

print(generated_text)

Shall I compare thee to a summer's day thou art more more more more


In [20]:
seed_text = 'Rough winds'
next_words = 15

generated_text = generate_text(
    seed_text,
    next_words,
    max_sequence_len
)
print(generated_text)

Rough winds winds shake the darling buds of may and summer's lease hath all too too a


In [ ]:
text = """Shall I compare thee to a summer's day? Thou art more lovely and more temperate.
Rough winds do shake the darling buds of May, And summer's lease hath all too short a date."""